# Ch.1 — Clustering

> **The story.** **Stuart Lloyd** invented k-means in **1957** inside Bell Labs while solving a pure engineering problem: how do you quantise a continuous audio signal into a finite codebook for PCM telephone transmission? His algorithm — assign each signal sample to the nearest codeword, then recompute codewords as means — was an internal Bell Labs technical report that stayed unpublished for **25 years**. By the time Lloyd finally published in **1982** the algorithm was already everywhere: **Hugo Steinhaus** had described it in 1956, **Edward Forgy** had rediscovered it in 1965, and **James MacQueen** had coined the name "k-means" in 1967. The density-based alternative arrived nearly four decades later: **Martin Ester, Hans-Peter Kriegel, Jörg Sander, and Xiaowei Xu** published DBSCAN at **KDD 1996**, winning the conference's Test-of-Time award in 2014. Clustering automates and scales the intuition to 440 wholesale customers and beyond.
>
> **Where you are in the curriculum.** You have just finished the Reinforcement Learning track (AgentAI), where every chapter had a reward signal telling the agent what was good. Now the signal disappears. This is the entry point to **unsupervised learning** — no labels, no target variable, no ground truth. The wholesale retailer wants to discover natural customer segments from purchase behaviour alone. Clustering is the first tool: group similar customers automatically, then name the segments afterward. It sets up dimensionality reduction in [Ch.2 →](../ch02_dimensionality_reduction) (PCA/t-SNE/UMAP to visualise 6D clusters in 2D) and the cluster-quality metrics in [Ch.3 →](../ch03_unsupervised_metrics) (silhouette, Davies-Bouldin — how do you score a clustering with no ground truth?).
>
> **Notation in this chapter.** $\mathbf{x}_i \in \mathbb{R}^d$ — a data point (one customer's spending vector, $d=6$ features); $K$ — number of clusters; $\boldsymbol{\mu}_k$ — centroid of cluster $k$; $C_k$ — set of points assigned to cluster $k$; $J = \sum_{k=1}^{K}\sum_{\mathbf{x}_i \in C_k}\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$ — inertia (K-Means objective); $\varepsilon$ — DBSCAN neighbourhood radius; $\text{minPts}$ — DBSCAN density threshold; $s(i)$ — silhouette coefficient of point $i \in [-1, 1]$.

---

## 0 · The Challenge

> **The mission**: Build **SegmentAI** — discover actionable customer segments from 440 wholesale customers, silhouette >0.5, satisfying 5 constraints.

**What we know so far:**
- Dataset: 440 wholesale customers, 6 spending features (Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicatessen)
- RL track complete — AgentAI achieved ≥195/200 CartPole steps
- **No labels — supervised learning is impossible here**
- **No baseline clusters yet — SegmentAI has not segmented a single customer**

**What's blocking us:**

The CMO asks: *"What types of customers do we have?"* No one has labelled these 440 customers as "loyalists" or "price-sensitive" — that taxonomy does not exist. Every supervised algorithm built so far needs a target column. We have none.

**What this chapter unlocks:**

Three clustering algorithms to discover structure without labels: K-Means (fast centroids), DBSCAN (arbitrary shapes + outlier detection), and HDBSCAN (hierarchy-based, no K required).

| Constraint | Target | Status after this chapter |
|-----------|--------|--------------------------|
| **#1 SEGMENTATION** | Silhouette >0.5 | k=4 found; silhouette=0.52 — above threshold |
| **#2 INTERPRETABILITY** | Business-actionable segment names | Centroids named: HoReCa, Retail, Mixed, Outlier |
| **#3 OUTLIER HANDLING** | Flag anomalous customers | DBSCAN labels extreme spenders as noise |
| **#4 NO LABELS** | Fully unsupervised | [Done] throughout |
| **#5 SCALABILITY** | 1M+ customers | K-Means O(nKd) per iteration |

## Core Idea

**K-Means:** Place $K$ imaginary "centre points" in the data, assign each customer to the nearest centre, then move each centre to the mean position of its customers. Repeat until nothing changes. The centres are called centroids. The goal is to minimise total within-cluster spread — customers should be close to their own centroid and far from others.

> **Optional depth:** The formal objective is $J = \sum_{k=1}^{K}\sum_{\mathbf{x}_i \in C_k}\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$ — the sum of squared Euclidean distances from each point to its assigned centroid. Lloyd's algorithm provably decreases $J$ at every step and converges in finite iterations, though not necessarily to the global minimum.

**DBSCAN:** Instead of pre-specifying $K$, define a "neighbourhood" — the region within radius $\varepsilon$ of a point. A customer is a *core customer* if at least `min_samples` others are within that neighbourhood. Clusters grow outward by density-reachability. Customers nobody can reach become **noise** (label $-1$). No K required; handles arbitrarily shaped clusters; isolates extreme outlier spenders automatically.

**HDBSCAN:** DBSCAN with one critical improvement — instead of fixing $\varepsilon$, it builds a complete hierarchy over all density levels and extracts the most *stable* clusters. You only need `min_cluster_size`. The algorithm finds clusters at whatever scale the data supports, rather than forcing a single global density threshold.

```
Algorithm    | K required? | Outlier label | Cluster shape   | Scales to 1M?
-------------|-------------|---------------|-----------------|---------------
K-Means      | Yes         | None (forced) | Spherical       | Yes (mini-batch)
DBSCAN       | No          | Noise  = -1   | Arbitrary       | With index
HDBSCAN      | No (cut)    | Noise  = -1   | Any             | Slower O(n log n)
```

## Running Example — SegmentAI

The CMO at a wholesale food distributor needs to stop treating all 440 customers identically. A hotel that orders bulk fresh produce has nothing in common with a corner shop restocking milk and groceries weekly — but without labels, the company has never formally defined "types." SegmentAI's first move is to let the data speak: run clustering on the 6 spending features, find the natural groups, and name them afterward.

Dataset: **UCI Wholesale Customers** — 440 customers, 6 spending features (Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicatessen). No target variable — pure unsupervised learning. After clustering you will colour customers by segment label to see if spending tiers, channel splits (hotel vs retail), or speciality patterns emerge.

**Predict:** Before running a single line of code — how many distinct customer types do you expect to find? Write your guess down. You'll check it against the elbow curve.

## How K-Means Works — The Lloyd Loop

Imagine scattering $K$ pins randomly into your customer data cloud. Each pin is a "centroid." Every customer walks to the nearest pin. Then every pin moves to the centre of its group. Repeat. The pins migrate until nobody switches groups anymore.

```
Iteration 0 — Random init        Iteration 1 — Assign          Iteration 2 — Recompute
  ·  ×  ·    ×                    ·  +  ·    ·                  ·  +  ·    ·
  ×  ·  ·    ·   assign →         ·  +  +    +   recompute →    ·  +  +    +
  ·  ·  ◆    ·                    ·  ·  ◆    ◆                  ·  ·  ◆    ◆
                                                                  ↑ centroids shifted

× = random centroid   + = assigned to ×   ◆ = assigned to ◆
After 2–20 iterations: centroids stop moving → convergence
```

```mermaid
flowchart TD
    A["Initialise K centroids\n(K-Means++ — smart spread)"] --> B["Assign each point\nto nearest centroid"]
    B --> C["Recompute each centroid\nas mean of assigned points"]
    C --> D{"Assignments\nchanged?"}
    D -- "Yes" --> B
    D -- "No" --> E["Converged\nReturn labels + centroids"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If you set K=440 (every customer is their own cluster), what is the inertia? What is the silhouette score? Think before implementing.

In [ ]:
# TODO: Implement this cell
#  (Setup)
#
# Steps:
# 1. Setup
# 2. Compute `IMG` using `Path()`
# 3. Load data
# 4. Fit the model -- call `transform()`
# 5. Process data
#
# Hint:
#    IMG = Path(???)
#    scaler = StandardScaler(???)
#    df = pd.read_csv(???)
#    X_log = np.log1p(???)

## §1 · K-Means: Finding the Right K

SegmentAI needs a concrete number of segments before it can build campaigns. Too few and the "hotel bulk buyer" gets lumped with the "corner shop." Too many and each campaign targets five customers. The elbow curve answers this: sweep K=2…10, measure how much tighter the clusters become, find where adding more clusters stops helping.

```
Elbow curve — inertia vs K (schematic):

Inertia
  |
  |  *                        ← K=2: two massive blobs
  |     *
  |        *  ← elbow at K=5  ← marginal gain flattens here
  |           *
  |            *   *   *   *  ← K=8+: tiny improvement, many segments
  +---+---+---+---+---+---+--- K
  2   3   4   5   6   7   8
```

Inertia always decreases — that is arithmetic, not a signal. The **elbow** is the signal. Silhouette score provides a second, independent check: it peaks at the K where cluster quality is geometrically best.

**Your turn:** Run the sweep below. Does the elbow match your earlier prediction?

In [ ]:
# TODO: Implement this cell
#  (K-Means elbow + silhouette sweep)
#
# Steps:
# 1. K-Means elbow + silhouette sweep
# 2. Fit the model -- call `append()`
# 3. Plot results -- call `subplots()`
# 4. Call `plot()` to produce the result
# 5. Plot results -- call `tight_layout()`
# 6. Compute `best_k_sil` using `argmax()`
#
# Hint:
#    km = KMeans(n_clusters=???, init=???)
#    km.fit(???)

## K-Means: Fit and Interpret Clusters

Fit with K=5 (business requirement: 5 actionable segments).
Inspect centroids in the **original** (un-scaled) feature space to understand what each segment represents.

In [ ]:
# TODO: Implement this cell
#  (Fit K-Means with K=5)
#
# Steps:
# 1. Fit K-Means with K=5
# 2. Compute `centroids_log` using `scale()`
# 3. Compute `segment_names`
# 4. Call `segments()` to produce the result
#
# Hint:
#    km_best = KMeans(n_clusters=???, init=???)
#    centroids_log = scaler.inverse_transform(???)
#    centroids_orig = np.expm1(???)
#    km_best.fit(???)

## K-Means: 2D Visualisation via PCA

We can't plot 6 dimensions directly. Use PCA to project to 2D and colour by cluster label.

In [ ]:
# TODO: Implement this cell
#  (PCA 2D projection)
#
# Steps:
# 1. PCA 2D projection
# 2. Plot results -- call `subplots()`
# 3. Plot results -- call `scatter()`
# 4. Plot results -- call `colour()`
# 5. Plot results -- call `tight_layout()`
#
# Hint:
#    pca2 = PCA(n_components=???, random_state=???)
#    X_2d = pca2.fit_transform(???)
#    axes = plt.subplots(???)
#    pca2.fit_transform(???)

## DBSCAN: ε Estimation via k-NN Distance Plot

**Rule of thumb:** `k = 2 × n_features = 12`.
Sort all customers by their distance to the k-th nearest neighbour.
The **knee** of the resulting curve is a good starting ε.

In [ ]:
# TODO: Implement this cell
#  (k-NN distance plot for ε estimation)
#
# Steps:
# 1. k-NN distance plot for ε estimation
# 2. Plot results -- call `subplots()`
# 3. Call `percentile()` to produce the result
#
# Hint:
#    nbrs = NearestNeighbors(n_neighbors=???, algorithm=???)
#    _ = nbrs.kneighbors(???)
#    knn_dists = np.sort(???)
#    ax = plt.subplots(???)

## DBSCAN: Fit and Inspect

In [ ]:
# TODO: Implement this cell
#  (DBSCAN)
#
# Steps:
# 1. DBSCAN
# 2. Fit the model -- call `DBSCAN()`
# 3. Compute `n_clusters_db` using `DBSCAN()`
# 4. Plot results -- call `scatter()`
# 5. Call `profiles()` to produce the result
#
# Hint:
#    db = DBSCAN(eps=???, min_samples=???)
#    ax = plt.subplots(???)
#    colours = labels_db.copy(???)
#    sc = ax.scatter(???)

## HDBSCAN

In [ ]:
# TODO: Implement this cell
#  (HDBSCAN)
#
# Steps:
# 1. HDBSCAN
# 2. Call `noise()` to produce the result
# 3. Plot results -- call `scatter()`
# 4. Process data
#
# Hint:
#    hdb = hdbscan.HDBSCAN(???)
#    labels_hdb = hdb.fit_predict(???)
#    ax = plt.subplots(???)
#    h_colours = labels_hdb.copy(???)

## Hyperparameter Dial: K-Means K Sweep

Visually compare how segment boundaries shift as K increases from 2 to 6.

In [ ]:
# TODO: Implement this cell
#  (K sweep visualisation)
#
# Steps:
# 1. K sweep visualisation
# 2. Plot results -- call `KMeans()`
# 3. Plot results -- call `suptitle()`
#
# Hint:
#    km_k = KMeans(n_clusters=???, init=???)
#    axes = plt.subplots(???)
#    km_k.fit(???)

## Hyperparameter Dial: DBSCAN ε Sweep

In [ ]:
# TODO: Implement this cell
#  (ε sweep visualisation)
#
# Steps:
# 1. ε sweep visualisation
# 2. Plot results -- call `scatter()`
# 3. Plot results -- call `suptitle()`
#
# Hint:
#    db_e = DBSCAN(eps=???, min_samples=???)
#    axes = plt.subplots(???)
#    cols = lbl.copy(???)
#    db_e.fit(???)

## What Can Go Wrong: K-Means on Non-Spherical Data

K-Means assumes clusters are convex and roughly equal-sized. On ring-shaped or elongated distributions it fails while DBSCAN succeeds.

In [ ]:
# TODO: Implement this cell
#  (Synthetic demo: K-Means vs DBSCAN on non-spherical shapes)
#
# Steps:
# 1. Synthetic demo: K-Means vs DBSCAN on non-spherical shapes
# 2. Call `seed()` to produce the result
# 3. Plot results -- call `subplots()`
# 4. Plot results -- call `KMeans()`
# 5. Plot results -- call `DBSCAN()`
# 6. Plot results -- call `suptitle()`
#
# Hint:
#    km_fail = KMeans(n_clusters=???, n_init=???)
#    db_ok = DBSCAN(eps=???, min_samples=???)
#    axes = plt.subplots(???)

## What Can Go Wrong: Unscaled Features

In [ ]:
# TODO: Implement this cell
#  (Scaling matters)
#
# Steps:
# 1. Scaling matters
# 2. Process data
# 3. Fit the model -- call `KMeans()`
#
# Hint:
#    km_raw = KMeans(n_clusters=???, n_init=???)
#    km_log = KMeans(n_clusters=???, n_init=???)

## Exercises

1. **DBSCAN noise analysis.** Examine the noise customers identified by DBSCAN. What makes them outliers? Compute their average spending per feature and compare to the cluster centroids. Are they extreme spenders or minimal spenders?

2. **K-Means++ vs random init.** Run `KMeans(n_clusters=5, init='random', n_init=1)` ten times with different seeds and record inertia. Then run `init='k-means++'`. Plot both inertia distributions as histograms. How much more stable is K-Means++?

3. **Segment radar chart.** Create a radar (spider) plot showing the centroid profile for each of the 5 segments. Which features most distinguish "Big Spenders" from "Price-Sensitive"?

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above